# Pipeline eval-data

`helpers.py` contient les fonctions complexes. Deux passes : les feeds
construisent modèles / benchmarks / scores, puis les métadonnées viennent se
poser dessus. Sortie : `database/{models,benchmarks,scores}.csv`.

Un modèle est identifié par le tuple `(base_model, reasoning_level,
reasoning_tokens, snapshot, variant)`, `''` = inconnu. 

## 0 — Setup

In [1]:
import json
import re

import pandas

from helpers import *  # Path, IDENT_COLS, load_aliases, resolve_model, split_name,
                       # model_id, benchmark_id, strip_name, seal_entries, dedup, report_leftovers

ROOT = Path.cwd()
FEED, METADATA, DATABASE = ROOT / "input/feed", ROOT / "input/metadata", ROOT / "database"

# feed -> comment traduire ses noms : "découpe" = aliases.csv d'abord, sinon découpe
# du nom machine (helpers.split_name) ; "énumération" = seul aliases.csv fait foi.
SOURCES = {"epoch": "découpe", "epoch_live": "découpe", "kaggle": "découpe",
           "rand": "énumération", "seal": "énumération"}
ALIASES = {s: load_aliases(FEED / s / "aliases.csv") for s in SOURCES}

MODEL_COLS = ["model_id", *IDENT_COLS, "organization", "country",
              "publication_date", "training_flop"]
BENCH_COLS = ["benchmark_id", "name", "category", "floor", "access"]
SCORE_COLS = ["model_id", "benchmark_id", "score", "stderr", "feed"]

models, benchmarks, scores = {}, {}, []
meta_obs = []                            # (model_id, date, FLOP) glanés au passage des feeds
unmapped = {s: set() for s in SOURCES}   # noms non résolus, par source
dropped = {s: set() for s in SOURCES}    # DROP explicites (alias à base_model vide)
new_names = []                           # un enregistrement par modèle créé
completed = []                           # efforts complétés depuis un nom affiché


def take(feed, raw, bench, score, stderr="", effort=""):
    """Traduit le nom, crée modèle/benchmark au besoin, empile le score. -> model_id ou None.

    `effort` (lu dans le `Name` affiché d'Epoch) ne complète qu'une identité sans effort
    connu ; variant='thinking' est exclu, son nom machine a déjà dit sa config.
    """
    status, ident = resolve_model(raw, ALIASES[feed], SOURCES[feed] == "découpe")
    if status != "ok":
        (unmapped if status == "unmapped" else dropped)[feed].add(strip_name(raw))
        return None
    if (effort and not ident["reasoning_level"] and not ident["reasoning_tokens"]
            and ident["variant"] != "thinking"):
        completed.append(ident["variant"] or "bare")
        ident = {**ident, "reasoning_level": effort,
                 "variant": "" if ident["variant"] == "unknown-effort" else ident["variant"]}
    mid = model_id(ident)
    if mid not in models:
        models[mid] = {"model_id": mid, **ident}
        new_names.append({"feed": feed, "raw_name": strip_name(raw), "model_id": mid,
                          "ident": tuple(ident[c] for c in IDENT_COLS)})
    bid = benchmark_id(bench)
    benchmarks.setdefault(bid, {"benchmark_id": bid, "name": bench})
    scores.append({"model_id": mid, "benchmark_id": bid, "score": float(score),
                   "stderr": "" if stderr == "" or pandas.isna(stderr) else float(stderr),
                   "feed": feed})
    return mid


def unit_range(feed):
    """Un score est une proportion ; tout le reste est une erreur de spec."""
    got = [r["score"] for r in scores if r["feed"] == feed]
    bad = [s for s in got if not 0 <= s <= 1]
    assert not bad, f"{feed}: {len(bad)} score(s) hors [0,1], ex. {bad[:3]}"
    print(f"{feed}: {len(got):,} scores, tous dans [0, 1]")

### La traduction d'un nom, sous les yeux

Chaîne d'appel : boucle du feed → `take(...)` (cellule Setup) → `helpers.resolve_model` →
le `aliases.csv` de la source d'abord, sinon `helpers.split_name` (la découpe du nom machine).
`''` = inconnu. Les exemples ci-dessous passent par ce chemin exact.

In [2]:
exemples = [("epoch",  "claude-opus-4-8_high"),                    # suffixe _high -> reasoning_level
            ("epoch",  "zai-org/glm-4.7"),                         # préfixe fournisseur retiré
            ("epoch",  "claude-opus-4-6_120K"),                    # budget _120K -> reasoning_tokens
            ("epoch",  "claude-opus-4-5-20251101_unknown"),        # date -> snapshot ; _unknown -> variant
            ("kaggle", "claude-opus-4-8"),                         # nom nu : tout reste inconnu
            ("seal",   "Opus 4.8"),                                # graphie libre : via aliases.csv
            ("rand",   "Claude 3.7 Sonnet V1 - Reasoning 16000")]  # énumération : via aliases.csv
lignes = []
for source, nom in exemples:
    statut, ident = resolve_model(nom, ALIASES[source], SOURCES[source] == "découpe")
    lignes.append({"source": source, "nom brut": nom, **(ident or {})})
pandas.DataFrame(lignes)

,source,nom brut,base_model,reasoning_level,reasoning_tokens,snapshot,variant
0,epoch,claude-opus-4-8_high,claude-opus-4-8,high,,,
1,epoch,zai-org/glm-4.7,glm-4.7,,,,
2,epoch,claude-opus-4-6_120K,claude-opus-4-6,,120000,,
3,epoch,claude-opus-4-5-20251101_unknown,claude-opus-4-5,,,2025-11-01,unknown-effort
4,kaggle,claude-opus-4-8,claude-opus-4-8,,,,
5,seal,Opus 4.8,claude-opus-4-8,,,,unknown-effort
6,rand,Claude 3.7 Sonnet V1 - Reasoning 16000,claude-3-7-sonnet,,16000,2025-02-19,


## 1 — Feeds

Chaque feed a **son dictionnaire** et **sa fonction de traduction** `translate_<feed>`
(comment amener ses données au format du projet). La **boucle unique**, en fin de section, fait pour
chaque feed :

```
charger le feed
traduire chaque donnée au format du projet      (translate_<feed>)
retrouver ou créer le modèle et le benchmark    (take -> aliases.csv, sinon découpe)
empiler le score avec les deux IDs
```



Une fonction de traduction rend des sextuplets `(nom brut, benchmark, score, stderr, effort, meta)` —
stderr/effort/meta vides quand la source ne les publie pas.

### 1a — Epoch (ZIP)

In [3]:
# Dictionnaire : fichier du ZIP -> (benchmark, colonne de score, diviseur).
FILE_SPEC = {
    'adversarial_nli_external.csv':       ('Adversarial NLI', 'Score', 1.0),
    'aider_polyglot_external.csv':        ('Aider Polyglot', 'Percent correct', 100.0),
    'algotune_external.csv':              ('AlgoTune', 'Score', 1.0),
    'apex_agents_external.csv':           ('APEX Agents', 'Pass@1 score', 1.0),
    'arc_agi_2_external.csv':             ('ARC-AGI-2', 'Score', 1.0),
    'arc_agi_external.csv':               ('ARC-AGI', 'Score', 1.0),
    'arc_ai2_external.csv':               ('ARC (AI2)', 'Challenge score', 1.0),
    'balrog_external.csv':                ('BALROG', 'Average progress', 1.0),
    'bbh_external.csv':                   ('BIG-Bench Hard (BBH)', 'Average', 1.0),
    'blueprint_bench_2_external.csv':     ('BlueprintBench 2', 'Score', 1.0),
    'bool_q_external.csv':                ('BoolQ', 'Score', 1.0),
    'cad_eval_external.csv':              ('CAD-Eval', 'Overall pass (%)', 1.0),
    'cl_bench_external.csv':              ('CL-Bench', 'Overall', 1.0),
    'cl_bench_life_external.csv':         ('CL-Bench Life', 'Overall', 1.0),
    'common_sense_qa_2_external.csv':     ('CSQA2', 'Score', 1.0),
    'critpt_external.csv':                ('CritPt', 'Accuracy', 1.0),
    'cursorbench_external.csv':           ('CursorBench', 'Score', 1.0),
    'cybench_external.csv':               ('Cybench', 'Unguided % Solved', 1.0),
    'deepresearchbench_external.csv':     ('DeepResearchBench', 'Average score', 1.0),
    'deepswe_external.csv':               ('DeepSWE', 'Pass@1', 1.0),
    'enigma_eval_external.csv':           ('EnigmaEval', 'Accuracy', 1.0),
    'exploitbench_external.csv':          ('ExploitBench', 'Mean capability', 1.0),
    'fictionlivebench_external.csv':      ('Fiction.LiveBench', '120k token score', 1.0),
    'forecastbench_external.csv':         ('ForecastBench', 'Overall score', 100.0),
    'frontiercode_external.csv':          ('FrontierCode', 'Main score', 1.0),
    'gbaeval_external.csv':               ('GBAEval', 'Overall score', 1.0),
    'gdp_pdf_external.csv':               ('GDP.pdf', 'GDP.pdf score', 1.0),
    'gdpval_external.csv':                ('GDPval', 'Win Rate (%)', 1.0),
    'geobench_external.csv':              ('GeoBench', 'ACW Country %', 1.0),
    'gsm8k_external.csv':                 ('GSM8K', 'EM', 1.0),
    'gso_external.csv':                   ('GSO-Bench', 'Score OPT@1', 1.0),
    'hella_swag_external.csv':            ('HellaSwag', 'Overall accuracy', 1.0),
    'hle_external.csv':                   ("Humanity's Last Exam", 'Accuracy', 1.0),
    'lambada_external.csv':               ('LAMBADA', 'Score', 1.0),
    'lech_mazur_writing_external.csv':    ('Lech Mazur Writing', 'Mean score', 10.0),
    'live_bench_external.csv':            ('LiveBench', 'Global average', 100.0),
    'metr_time_horizons_external.csv':    ('METR Time Horizons', 'average_score', 1.0),
    'mindcube_external.csv':              ('MindCube', 'Overall score', 1.0),
    'mmlu_external.csv':                  ('MMLU', 'EM', 1.0),
    'open_book_qa_external.csv':          ('OpenBookQA', 'Accuracy', 1.0),
    'os_world_external.csv':              ('OS World (Screenshot)', 'Score', 100.0),
    'osworld_2_external.csv':             ('OS World 2', 'Binary accuracy', 1.0),
    'piqa_external.csv':                  ('PIQA', 'Score', 1.0),
    'posttrainbench_external.csv':        ('PostTrainBench', 'Average (%)', 1.0),
    'proofbench_external.csv':            ('ProofBench', 'Accuracy', 1.0),
    'rli_external.csv':                   ('Remote Labor Index', 'Score', 1.0),
    'scicode_external.csv':               ('SciCode', 'Score', 1.0),
    'science_qa_external.csv':            ('ScienceQA', 'Score', 1.0),
    'simplebench_external.csv':           ('SimpleBench', 'Score (AVG@5)', 1.0),
    'spatialviz_bench_external.csv':      ('SpatialViz-Bench', 'Overall score', 1.0),
    'superglue_external.csv':             ('SuperGLUE', 'Score', 1.0),
    'surface_evolver_bench_external.csv': ('Surface Evolver Bench', 'Mean score', 1.0),
    'terminalbench_external.csv':         ('TerminalBench', 'Accuracy mean', 1.0),
    'the_agent_company_external.csv':     ('The Agent Company', '% Score', 1.0),
    'trivia_qa_external.csv':             ('TriviaQA', 'EM', 1.0),
    'video_mme_external.csv':             ('Video-MME', 'Overall (no subtitles)', 1.0),
    'vpct_external.csv':                  ('VPCT', 'Correct', 1.0),
    'weirdml_external.csv':               ('WeirdML', 'Accuracy', 1.0),
    'wino_grande_external.csv':           ('WinoGrande', 'Accuracy', 1.0),
}

# Fichiers écartés, avec raison.
FILE_SKIP = {
    'ale_bench_external.csv':                    'points de compétition (~138-2041), pas une proportion [0,1]',
    'btf3_external.csv':                         'perte Brier/RPS agrégée, sans baseline « réponse au hasard » publiée : plancher non dérivable',
    'chess_puzzles.csv':                         'éval interne Epoch ; ingérée via le feed live (§1b)',
    'eci_benchmark_difficulties_and_slopes.csv': 'paramètres IRT ajustés par Epoch, en aval de cette pipeline',
    'epoch_capabilities_index.csv':              "l'ECI d'Epoch : un output en aval de cette pipeline, pas un benchmark",
    'frontiermath.csv':                          "jeu de problèmes v1 défectueux (42 % d'items corrigés en v2) ; le feed live (§1b) fournit v2",
    'frontiermath_tier_4.csv':                   "jeu de problèmes v1 défectueux (42 % d'items corrigés en v2) ; le feed live (§1b) fournit v2",
    'frontierswe_external.csv':                  'rang déguisé (corr. 0.977 avec le rang du board) : dépend des autres modèles, pas du modèle seul',
    'gpqa_diamond.csv':                          'éval interne Epoch ; ingérée via le feed live (§1b)',
    'math_level_5.csv':                          'éval interne Epoch ; ingérée via le feed live (§1b)',
    'mystery_game_puzzles.csv':                  'éval interne Epoch ; ingérée via le feed live (§1b)',
    'otis_mock_aime_2024_2025.csv':              'éval interne Epoch ; ingérée via le feed live (§1b)',
    'README.md':                                 'pas un fichier de données',
    'simpleqa_verified.csv':                     'éval interne Epoch ; ingérée via le feed live (§1b)',
    'swe_bench_verified.csv':                    'éval interne Epoch ; ingérée via le feed live (§1b)',
    'vending_bench_2_external.csv':              '3 lignes informatives seulement, et un score en dollars non comparable',
    'webdev_arena_external.csv':                 "score Elo d'arène, pas une proportion [0,1]",
}

# Colonnes pas déjà en proportion [0,1] : rescaling monotone contre une référence exogène.
FILE_SCORE_TRANSFORM = {"algotune_external.csv": (lambda x: 1.0 - 1.0 / x, "1 - 1/speedup")}

_both = sorted(set(FILE_SPEC) & set(FILE_SKIP))
if _both:
    raise ValueError(f"dans FILE_SPEC ET FILE_SKIP : {_both}")

In [4]:
# Traduction Epoch : un fichier = un benchmark.
EFFORT_RE = r"(?i)\((?:thinking,\s*)?(minimal|low|medium|high|x-high|xhigh|max|none)\)\s*$"  # effort parfois caché dans le nom affiché `Name`


def translate_epoch():
    unrecognized, blank, skipped = [], 0, 0
    for path in sorted((FEED / "epoch").glob("*.csv")):       # chaque fichier du feed
        if path.name == "aliases.csv":                        # le dictionnaire n'est pas une donnée
            continue
        if path.name in FILE_SKIP:                            # fichier écarté (raison dans FILE_SKIP)
            skipped += 1
            continue
        if path.name not in FILE_SPEC:                        # fichier inconnu : signalé, jamais ingéré
            unrecognized.append(path.name)
            continue
        bench, score_col, divisor = FILE_SPEC[path.name]      # benchmark + colonne de score, via le dictionnaire
        df = pandas.read_csv(path, low_memory=False)          # chargement
        score = pandas.to_numeric(df[score_col], errors="coerce") / divisor  # score -> proportion [0,1]
        if path.name in FILE_SCORE_TRANSFORM:                 # rescaling spécial (AlgoTune)
            score = FILE_SCORE_TRANSFORM[path.name][0](score)
        se = next((df[c] for c in (f"{score_col} Standard Error", "stderr") if c in df.columns), None)  # incertitude, si publiée
        effort = (df["Name"].astype(str).str.extract(EFFORT_RE, expand=False).str.lower()
                  .str.replace("x-high", "xhigh", regex=False).fillna("")
                  if "Name" in df.columns else pandas.Series("", index=df.index))  # effort lu dans `Name`
        meta_cols = {k: df[c] for k, c in (("publication_date", "Release date"),
                                           ("training_flop", "Training compute (FLOP)")) if c in df.columns}  # pour la passe 2
        for i, raw in df["Model version"].items():            # chaque ligne du fichier
            if pandas.isna(score[i]):                         # pas de score : rien à garder
                continue
            if pandas.isna(raw):                              # score sans nom de modèle : écarté et compté
                blank += 1
                continue
            yield (raw, bench, score[i], se[i] if se is not None else "",
                   effort[i], {k: v[i] for k, v in meta_cols.items()})  # une donnée au format du projet
    print(f"    {blank} ligne(s) écartées faute de `Model version`, {skipped} fichier(s) écartés")
    print(f"    efforts complétés depuis `Name` : {({v: completed.count(v) for v in set(completed)} or 0)}")
    for name in unrecognized:
        print(f"    ? {name} : ni dans FILE_SPEC ni dans FILE_SKIP")

### 1b — Epoch (feed live)

In [5]:
# Dictionnaire : tâche du feed live -> (benchmark, préfixe de `task version` exigé ou None).
# Toutes les évals internes d'Epoch passent par ici (une seule voie, pas de double comptage).
# FrontierMath v1/v2 : benchmarks SÉPARÉS, le jeu de problèmes v1 était défectueux.
LIVE_TASK_SPEC = {
    'Chess Puzzles':                          ('Chess Puzzles', None),
    'EBR-bench':                              ('EBR-bench', None),
    'FrontierMath-2025-02-28-Private':        ('FrontierMath v1', '1.'),
    'FrontierMath-Tier-4-2025-07-01-Private': ('FrontierMath Tier 4 v1', '1.'),
    'FrontierMath-Tier-4-v2-Private':         ('FrontierMath Tier 4', '2.'),
    'FrontierMath-Tiers-1-3-v2-Private':      ('FrontierMath', '2.'),
    'GPQA diamond':                           ('GPQA Diamond', None),
    'MATH level 5':                           ('MATH Level 5', None),
    'Mystery Game Puzzles':                   ('Mystery Game Puzzles', None),
    'OTIS Mock AIME 2024-2025':               ('OTIS Mock AIME 2024-2025', None),
    'SimpleQA Verified':                      ('SimpleQA Verified', None),
    'SWE-Bench verified':                     ('SWE-Bench Verified', None),
}

# Tâches écartées, avec raison.
LIVE_TASK_SKIP = {
    'FrontierMath-2025-02-28-Public':        "split public de 10 problèmes ; l'information serait fabriquée par le clip aux bornes",
    'FrontierMath-Tier-4-2025-07-01-Public': 'split public de 2 problèmes ; 3 valeurs distinctes seulement',
}

assert not set(LIVE_TASK_SPEC) & set(LIVE_TASK_SKIP), "tâche dans SPEC et SKIP"

In [6]:
# Traduction feed live : une tâche = un benchmark ; garde-fou sur la version du jeu de problèmes.
def translate_epoch_live():
    live = pandas.read_csv(FEED / "epoch_live/epoch_live_benchmarks.csv", low_memory=False)  # chargement
    for task, sub in live.groupby("task"):                    # chaque tâche du feed
        if task in LIVE_TASK_SKIP:                            # tâche écartée
            continue
        if task not in LIVE_TASK_SPEC:                        # tâche inconnue : signalée, jamais ingérée
            print(f"    ? tâche {task!r} : ni dans LIVE_TASK_SPEC ni dans LIVE_TASK_SKIP")
            continue
        bench, ver_prefix = LIVE_TASK_SPEC[task]              # benchmark, via le dictionnaire
        ver = sub["task version"].dropna().astype(str)        # garde-fou : bon jeu de problèmes ?
        bad = sorted(set(ver[~ver.str.startswith(ver_prefix)])) if ver_prefix else []
        if bad:                                               # mauvaise version : on s'arrête net
            raise ValueError(f"{task}: version(s) de problème inattendue(s) {bad}")
        score = pandas.to_numeric(sub["mean_score"], errors="coerce")  # score, déjà en [0,1]
        stderr = pandas.to_numeric(sub["stderr"], errors="coerce")     # incertitude publiée
        for i, raw in sub["model"].items():                   # chaque ligne = un modèle
            if pandas.isna(raw) or pandas.isna(score[i]):     # pas de nom ou pas de score
                continue
            yield raw, bench, score[i], stderr[i], "", {"publication_date": sub["Version release date"][i]}

### 1c — Kaggle Open Benchmarks

In [7]:
# Dictionnaire : board -> (benchmark, kaggle_wins).
# kaggle_wins : en cas de désaccord (modèle, benchmark), la ligne Kaggle gagne (lu par la dédup, §2).
KAGGLE_SPEC = {
    'deepmind/simpleqa-verified': ('SimpleQA Verified', False),
    'open-benchmarks/mmlu':       ('MMLU', True),
    'open-benchmarks/mmlu-pro':   ('MMLU-Pro', True),
}

In [8]:
# Traduction Kaggle : un board = un benchmark ; seul nettoyage de nom, le tag « -default ».
def translate_kaggle():
    for path in sorted((FEED / "kaggle").glob("*.json")):     # fichier inconnu : signalé
        if path.stem.replace("_", "/", 1) not in KAGGLE_SPEC:
            print(f"    ? {path.name} : pas dans KAGGLE_SPEC")
    for board, (bench, _) in KAGGLE_SPEC.items():             # chaque board du dictionnaire
        payload = json.loads((FEED / "kaggle" / (board.replace("/", "_") + ".json")).read_text())  # chargement
        for row in payload.get("rows", []):                   # chaque ligne = un modèle
            raw = re.sub(r"-default$", "", row.get("modelVersionSlug", ""))  # retire le tag de harness
            for tr in row.get("taskResults", []):             # chaque résultat de tâche
                num = (tr.get("result") or {}).get("numericResult") or {}
                if "value" not in num:                        # pas de score : rien à garder
                    continue
                ci = num["confidenceInterval"] if num.get("hasConfidenceInterval") else ""  # demi-largeur d'IC 95 % -> écart-type (convention reprise de l'ancien pipeline)
                yield raw, bench, num["value"], ci / 1.96 if ci != "" else "", "", {}

### 1d — RAND (bio / chimie)

In [9]:
# Traduction RAND : format large, une colonne par benchmark (« Bio: X », « Chem: Y »).
def translate_rand():
    rand = pandas.read_csv(FEED / "rand/rand_benchmark_scores.csv")  # chargement
    long = rand.iloc[2:].melt(id_vars=["Model"], var_name="col", value_name="brut")  # 2 premières lignes = baseline humaine, écartées ; large -> long
    for row in long.itertuples():                             # chaque case = (modèle, benchmark)
        score = pandas.to_numeric(row.brut, errors="coerce")  # 'N/A' -> écarté
        if pandas.isna(score):
            continue
        yield row.Model, row.col.split(": ", 1)[-1], score / 100, "", "", {}  # benchmark = colonne sans préfixe

### 1e — Scale SEAL

In [10]:
# Dictionnaire : identifiant de board -> benchmark. Scores 0-100, pas de stderr publié.
# rli est AUSSI dans le ZIP : recouvrement partiel voulu, la dédup tranche.
SEAL_SPEC = {
    'audiomc':                       'AudioMultiChallenge',
    'instruction_following':         'SEAL Instruction Following',
    'mcp_atlas':                     'MCP Atlas',
    'multichallenge':                'MultiChallenge',
    'multinrc':                      'MultiNRC',
    'prbench-finance':               'PRBench Finance',
    'prbench-legal':                 'PRBench Legal',
    'rli':                           'Remote Labor Index',
    'swe_bench_pro_private':         'SWE-Bench Pro (Private)',
    'swe_bench_pro_public':          'SWE-Bench Pro',
    'tool_use_enterprise':           'SEAL Tool Use (Enterprise)',
    'tutorbench':                    'TutorBench',
    'visual_language_understanding': 'Visual Task Assessment (VISTA)',
    'vtb':                           'VisualToolBench',
}

# Boards écartés, avec raison.
SEAL_SKIP = {
    'arabic':                        'score Elo multilingue, pas une proportion [0,1]',
    'chinese':                       'score Elo multilingue, pas une proportion [0,1]',
    'coding':                        'score Elo (~600-1240), pas une proportion [0,1]',
    'humanitys_last_exam_text_only': 'sous-ensemble texte seul ; pas comparable au HLE complet ingéré via Epoch',
    'japanese':                      'score Elo multilingue, pas une proportion [0,1]',
    'spanish':                       'score Elo multilingue, pas une proportion [0,1]',
}

assert not set(SEAL_SPEC) & set(SEAL_SKIP), "board dans SPEC et SKIP"

In [11]:
# Traduction SEAL : un board = un benchmark, une page HTML capturée par board.
def translate_seal():
    pages = sorted((FEED / "seal").glob("*.html"))
    if not pages:            # pages non redistribuées (licence amont, cf. NOTICE.md)
        raise FileNotFoundError(
            "input/feed/seal/ ne contient aucune page HTML. Les pages Scale SEAL ne sont "
            "pas redistribuées avec le dépôt (voir NOTICE.md) : lancer `python3 refresh.py` "
            "avant le Run All.")
    for path in pages:                                        # chaque page capturée
        if path.stem in SEAL_SKIP:                            # board écarté
            continue
        if path.stem not in SEAL_SPEC:                        # board inconnu : signalé, jamais ingéré
            print(f"    ? {path.name} : ni dans SEAL_SPEC ni dans SEAL_SKIP")
            continue
        entries = seal_entries(path.read_text())              # extrait le tableau des scores de la page
        if not entries:                                       # page sans tableau : signalée
            print(f"    ! {path.name} : aucun tableau `entries` dans la page")
            continue
        for e in entries:                                     # chaque ligne = un modèle
            if isinstance(e.get("score"), (int, float)):      # score présent et numérique ?
                yield e["model"], SEAL_SPEC[path.stem], e["score"] / 100, "", "", {}  # scores publiés sur 100

### La boucle — pour chaque feed : charger, traduire, insérer

In [12]:
TRANSLATE = {"epoch": translate_epoch, "epoch_live": translate_epoch_live,
             "kaggle": translate_kaggle, "rand": translate_rand, "seal": translate_seal}
assert TRANSLATE.keys() == SOURCES.keys(), "chaque feed doit avoir sa fonction de traduction"

for feed in SOURCES:                                          # pour chaque feed
    print(f"{feed}:")
    n0 = len(scores)
    for nom, bench, score, stderr, effort, meta in TRANSLATE[feed]():  # charger + traduire
        mid = take(feed, nom, bench, score, stderr, effort)   # nom -> colonnes ; créer si nouveau ; empiler le score
        if mid and meta:                                      # métadonnées glanées, pour la passe 2
            meta_obs.append({"model_id": mid, **meta})
    unit_range(feed)                                          # garde-fou : que des proportions [0,1]

drops = {f: sorted(d) for f, d in dropped.items() if d}
print(f"\nDROP explicites : {drops or 'aucun'}")

epoch:
    416 ligne(s) écartées faute de `Model version`, 16 fichier(s) écartés
    efforts complétés depuis `Name` : {'unknown-effort': 12, 'bare': 29}
    ? mirrorcode.csv : ni dans FILE_SPEC ni dans FILE_SKIP
epoch: 3,854 scores, tous dans [0, 1]
epoch_live:
    ? tâche 'MirrorCode' : ni dans LIVE_TASK_SPEC ni dans LIVE_TASK_SKIP
    ? tâche 'OEIS Open' : ni dans LIVE_TASK_SPEC ni dans LIVE_TASK_SKIP
    ? tâche 'OEIS Open Lite' : ni dans LIVE_TASK_SPEC ni dans LIVE_TASK_SKIP
    ? tâche 'SimpleQA Verified (anti-abstention)' : ni dans LIVE_TASK_SPEC ni dans LIVE_TASK_SKIP
epoch_live: 1,346 scores, tous dans [0, 1]
kaggle:
kaggle: 154 scores, tous dans [0, 1]
rand:
rand: 550 scores, tous dans [0, 1]
seal:
seal: 418 scores, tous dans [0, 1]

DROP explicites : {'rand': ['Unsafety Llama']}


## 2 — Dédoublonnage

Même (modèle, benchmark) vu deux fois : d'abord les doublons exacts (deux
transports du même chiffre), puis les désaccords — le board Kaggle gagne là où
`kaggle_wins`, sinon le score max.

In [13]:
scores_df = pandas.DataFrame(scores, columns=SCORE_COLS)
models_df = pandas.DataFrame(models.values())
benchmarks_df = pandas.DataFrame(benchmarks.values())

kaggle_win_ids = {benchmark_id(b) for b, wins in KAGGLE_SPEC.values() if wins}
deduped, dups, n_exact = dedup(scores_df, kaggle_win_ids)

print(f"scores avant {len(scores_df):,} -> après {len(deduped):,} "
      f"(delta {len(scores_df) - len(deduped):,} = {n_exact:,} doublons exacts "
      f"+ {int(dups['n'].sum() - len(dups)) if len(dups) else 0:,} désaccords)")
print(f"{len(dups):,} paire(s) (modèle, benchmark) en désaccord")
if len(dups):
    _b = benchmarks_df.set_index("benchmark_id")["name"]
    _m = models_df.set_index("model_id")["base_model"]
    print(dups.head(12)
          .assign(modele=lambda d: d["model_id"].map(_m), benchmark=lambda d: d["benchmark_id"].map(_b))
          [["modele", "benchmark", "n", "score_min", "score_max", "spread", "feeds"]]
          .to_string(index=False))

scores avant 6,322 -> après 5,437 (delta 885 = 316 doublons exacts + 569 désaccords)
318 paire(s) (modèle, benchmark) en désaccord
                     modele            benchmark  n  score_min  score_max  spread feeds
             Llama-2-70b-hf                GSM8K  5     0.1330     0.6960  0.5630 epoch
gemini-3-deep-think-preview            ARC-AGI-2  2     0.4514     0.8458  0.3944 epoch
   Meta-Llama-3-8B-Instruct           OpenBookQA  2     0.4500     0.8260  0.3760 epoch
           Llama-2-13b-chat                GSM8K  2     0.0270     0.3690  0.3420 epoch
                  PaLM 540B            ARC (AI2)  3     0.5300     0.8520  0.3220 epoch
        Baichuan-2-13B-Base                GSM8K  2     0.2210     0.5280  0.3070 epoch
                   LLaMA-7B           OpenBookQA  3     0.2840     0.5720  0.2880 epoch
           text-davinci-003                GSM8K  3     0.5060     0.7820  0.2760 epoch
          Mixtral-8x7B-v0.1            ARC (AI2)  2     0.5970     0.8730  0.

## 3 — Métadonnées (passe 2)

Chaque source imprime ses deux restes : les lignes de métadonnées qui ne
correspondent à rien, et les modèles / benchmarks laissés sans métadonnée.

### 3a — organizations.csv -> organization, country

In [14]:
orgs = pandas.read_csv(METADATA / "organizations.csv").fillna("")
assert not orgs["base_model"].duplicated().any(), "base_model dupliqué : la jointure ferait fan-out"

m = models_df.merge(orgs, on="base_model", how="left", indicator=True)
report_leftovers("organizations.csv",
                 set(orgs["base_model"]) - set(models_df["base_model"]),
                 set(m.loc[m["_merge"] == "left_only", "base_model"]))
models_df = m.drop(columns="_merge").fillna({"organization": "", "country": ""})

organizations.csv: 15 ligne(s) meta sans cible, 42 cible(s) sans meta
    meta orpheline : GLM-4.5-FP8
    meta orpheline : Gemma-3-27b-It
    meta orpheline : QwQ-32B (16K thinking)
    meta orpheline : Qwen3-14B
    meta orpheline : Qwen3-235B-A22B
    meta orpheline : Qwen3-8B
    meta orpheline : codestral-2508
    meta orpheline : gemini-2.5-flash-preview-04-17 (16K thinking)
    sans meta      : ChatGPT agent
    sans meta      : Kimi-Audio-7B-Instruct
    sans meta      : LFM2-Audio-1.5B
    sans meta      : Manus 1.0
    sans meta      : Manus 1.5
    sans meta      : Manus_1.6
    sans meta      : MiMo-Audio-7B-Instruct
    sans meta      : Phi-4-multimodal-instruct


### 3b — dates et FLOP glanés chez Epoch -> publication_date, training_flop

In [15]:
# Un modèle apparaît dans des dizaines de fichiers ; Epoch n'y est pas toujours
# d'accord avec lui-même. On prend la valeur modale, et à égalité la plus petite
# (donc la date la plus ancienne).
obs = pandas.DataFrame(meta_obs)


def modal(col):
    v = obs[["model_id", col]].dropna()
    v = v[v[col].astype(str).str.strip() != ""]
    return (v.value_counts().reset_index(name="n")
             .sort_values(["n", col], ascending=[False, True])
             .drop_duplicates("model_id").set_index("model_id")[col])


for col in ("publication_date", "training_flop"):
    models_df[col] = models_df["model_id"].map(modal(col))

_named = models_df.set_index("model_id")["base_model"]
report_leftovers("epoch (dates/FLOP glanés)",
                 set(obs["model_id"]) - set(models_df["model_id"]),
                 set(_named[models_df.loc[models_df["publication_date"].isna(), "model_id"]]))
print(f"    dont sans FLOP : {int(models_df['training_flop'].isna().sum())} modèle(s)")
models_df = models_df.fillna({"publication_date": "", "training_flop": ""})

epoch (dates/FLOP glanés): 0 ligne(s) meta sans cible, 93 cible(s) sans meta
    sans meta      : ChatGPT agent
    sans meta      : Inkling
    sans meta      : Kimi-Audio-7B-Instruct
    sans meta      : LFM2-Audio-1.5B
    sans meta      : Llama-3.1-405B-Instruct
    sans meta      : Manus 1.0
    sans meta      : Manus 1.5
    sans meta      : Manus_1.6
    dont sans FLOP : 699 modèle(s)


### 3c — categories.csv -> category

In [16]:
cats = pandas.read_csv(METADATA / "categories.csv").fillna("")
assert not cats["benchmark"].duplicated().any(), "benchmark dupliqué dans categories.csv"

b = benchmarks_df.merge(cats[["benchmark", "category"]].rename(columns={"benchmark": "name"}),
                        on="name", how="left", indicator=True)
report_leftovers("categories.csv",
                 set(cats["benchmark"]) - set(benchmarks_df["name"]),
                 set(b.loc[b["_merge"] == "left_only", "name"]))
benchmarks_df = b.drop(columns="_merge").fillna({"category": ""})

categories.csv: 4 ligne(s) meta sans cible, 0 cible(s) sans meta
    meta orpheline : OS World v1 (A11y tree)
    meta orpheline : OS World v1 (Screenshot)
    meta orpheline : SWE-Bench Bash Only
    meta orpheline : Vending-Bench 2


### 3d — floors.csv -> floor

In [17]:
floors = pandas.read_csv(METADATA / "floors.csv").fillna("")
assert not floors["benchmark"].duplicated().any(), "benchmark dupliqué dans floors.csv"

b = benchmarks_df.merge(floors[["benchmark", "lower_bound"]]
                        .rename(columns={"benchmark": "name", "lower_bound": "floor"}),
                        on="name", how="left", indicator=True)
report_leftovers("floors.csv",
                 set(floors["benchmark"]) - set(benchmarks_df["name"]),
                 set(b.loc[b["_merge"] == "left_only", "name"]))
benchmarks_df = b.drop(columns="_merge").fillna({"floor": ""})

floors.csv: 10 ligne(s) meta sans cible, 0 cible(s) sans meta
    meta orpheline : AISI CTF Suites - Apprentice
    meta orpheline : AISI CTF Suites - Expert
    meta orpheline : AISI Cyber Ranges - The Last Ones
    meta orpheline : CTI-REALM
    meta orpheline : CyberGym
    meta orpheline : ExploitGym
    meta orpheline : OpenAI Cyber Ranges
    meta orpheline : OpenAI NYU CTF Professional (Original)


### 3e — access.csv -> access

In [18]:
access = pandas.read_csv(METADATA / "access.csv").fillna("")
assert not access["benchmark"].duplicated().any(), "benchmark dupliqué dans access.csv"

b = benchmarks_df.merge(access[["benchmark", "access"]].rename(columns={"benchmark": "name"}),
                        on="name", how="left", indicator=True)
report_leftovers("access.csv",
                 set(access["benchmark"]) - set(benchmarks_df["name"]),
                 set(b.loc[b["_merge"] == "left_only", "name"]))
benchmarks_df = b.drop(columns="_merge").fillna({"access": ""})

access.csv: 0 ligne(s) meta sans cible, 0 cible(s) sans meta


## 4 — Vérifications

In [19]:
assert not models_df.duplicated(IDENT_COLS).any(), "tuple d'identification dupliqué"
assert not benchmarks_df.duplicated("name").any(), "nom de benchmark dupliqué"
assert not deduped.duplicated(["model_id", "benchmark_id"]).any(), "score dupliqué"
assert set(deduped["model_id"]) <= set(models_df["model_id"]), "model_id orphelin"
assert set(deduped["benchmark_id"]) <= set(benchmarks_df["benchmark_id"]), "benchmark_id orphelin"
assert deduped["score"].between(0, 1).all(), "score hors [0,1]"
print(f"OK — {len(models_df):,} modèles, {len(benchmarks_df):,} benchmarks, {len(deduped):,} scores")

for feed, names in unmapped.items():
    print(f"non résolus [{feed}] : {sorted(names) if names else 'aucun'}")

prev_path = DATABASE / "models.csv"
if prev_path.exists() and prev_path.stat().st_size:
    prev = set(pandas.read_csv(prev_path)["model_id"])
    fresh = [n for n in new_names if n["model_id"] not in prev]
    print(f"\nnouveaux modèles vs database/models.csv : {len(fresh)}")
    for n in fresh[:20]:
        print(f"  {n['feed']:11} {n['raw_name']:45} {n['ident']}")
else:
    print(f"\npas de database/models.csv précédent : les {len(models_df):,} modèles sont tous nouveaux")

OK — 930 modèles, 100 benchmarks, 5,437 scores
non résolus [epoch] : aucun
non résolus [epoch_live] : aucun
non résolus [kaggle] : aucun
non résolus [rand] : aucun
non résolus [seal] : ['Gemini 3.7 Flash']

nouveaux modèles vs database/models.csv : 0


## 5 — Écriture

In [20]:
DATABASE.mkdir(exist_ok=True)
for name, df, order in (
    ("models.csv", models_df[MODEL_COLS], IDENT_COLS),
    ("benchmarks.csv", benchmarks_df[BENCH_COLS], ["name"]),
    ("scores.csv", deduped[SCORE_COLS], ["benchmark_id", "model_id", "feed"]),
):
    out = df.sort_values(order).fillna("")
    out.to_csv(DATABASE / name, index=False, encoding="utf-8")
    print(f"{name}: {len(out):,} lignes x {len(out.columns)} colonnes")

models.csv: 930 lignes x 10 colonnes
benchmarks.csv: 100 lignes x 5 colonnes
scores.csv: 5,437 lignes x 5 colonnes


## 6 — Test de régression

`tests/expected_database/` est la copie bénie de `database/`. Un écart = soit une régression,
soit un changement voulu (nouveau feed, alias corrigé) — dans ce cas, re-bénir :
`cp database/*.csv tests/expected_database/`.

In [21]:
EXPECTED = ROOT / "tests/expected_database"
for f in ["models.csv", "benchmarks.csv", "scores.csv"]:
    now = pandas.read_csv(DATABASE / f, dtype=str).fillna("")   # la base qu'on vient d'écrire
    ref = pandas.read_csv(EXPECTED / f, dtype=str).fillna("")   # la base attendue
    if now.equals(ref):
        print(f"OK {f} : identique à la base attendue ({len(now):,} lignes)")
    else:                                                       # écart : colonnes puis lignes
        cols_plus, cols_moins = set(now) - set(ref), set(ref) - set(now)
        communes = sorted(set(now) & set(ref))
        nouvelles = len(now[communes].merge(ref[communes], how="left", indicator=True).query("_merge == 'left_only'"))
        disparues = len(ref[communes].merge(now[communes], how="left", indicator=True).query("_merge == 'left_only'"))
        print(f"ECART {f} : colonnes +{sorted(cols_plus) or 'aucune'} -{sorted(cols_moins) or 'aucune'}, "
              f"{nouvelles} ligne(s) nouvelle(s), {disparues} disparue(s) "
              f"— voulu ? alors : cp database/*.csv tests/expected_database/")

OK models.csv : identique à la base attendue (930 lignes)
OK benchmarks.csv : identique à la base attendue (100 lignes)
OK scores.csv : identique à la base attendue (5,437 lignes)
